# Module: Fetching Census Data
## US Census Bureau API - Tehama County, CA
**Data:** API pulls real Demographic and Sociaeconomic Data for Tehama County (FIPS:06103)
**Data Source:** ACS 5-Year Estimates
**Comparision:** Tehama Vs CA State Average
**Author:** Abishek Heyer
**Date:** April 2026

In [3]:
#Libraries
import requests
import pandas as pd
import os
from dotenv import load_dotenv

In [4]:
load_dotenv()
CENSUS_API_KEY = os.getenv("CENSUS_API_KEY")


In [5]:
# Key variables
TEHAMA_FIPS ="06103" #Tehama County
STATE_FIPS= "06"     #CA
COUNTY_FIPS= "103"   #Tehama within CA
ACS_YEAR = 2024      #Year 

if CENSUS_API_KEY:
    print("Key loaded Successfully")
else:
    prnt("Key not found")

Key loaded Successfully


In [6]:
# Variables & Constants 

VARIABLES = {

    # Age & Sex
    "B01003_001E": "total_population",
    "B01001_002E": "male_population",
    "B01001_026E": "female_population",
    "B01002_001E": "median_age",
    "B01001_003E": "male_under_5",
    "B01001_027E": "female_under_5",
    "B01001_004E": "male_5_to_9",
    "B01001_005E": "male_10_to_14",
    "B01001_006E": "male_15_to_17",
    "B01001_028E": "female_5_to_9",
    "B01001_029E": "female_10_to_14",
    "B01001_030E": "female_15_to_17",
    "B01001_020E": "male_65_66",
    "B01001_021E": "male_67_69",
    "B01001_022E": "male_70_74",
    "B01001_023E": "male_75_79",
    "B01001_024E": "male_80_84",
    "B01001_025E": "male_85_plus",
    "B01001_044E": "female_65_66",
    "B01001_045E": "female_67_69",
    "B01001_046E": "female_70_74",
    "B01001_047E": "female_75_79",
    "B01001_048E": "female_80_84",
    "B01001_049E": "female_85_plus",

    # Race & Ethnicity
    "B03003_003E": "hispanic_latino",
    "B02001_002E": "white_alone",
    "B02001_003E": "black_alone",
    "B02001_004E": "aian_alone",
    "B02001_005E": "asian_alone",
    "B02001_006E": "nhpi_alone",
    "B02001_008E": "two_or_more_races",

    # Income & Poverty
    "B19013_001E": "median_household_income",
    "B19301_001E": "per_capita_income",
    "B17001_002E": "population_below_poverty",
    "B17001_001E": "poverty_universe",

    # Housing
    "B25001_001E": "housing_units",
    "B25003_002E": "owner_occupied_units",
    "B25003_001E": "occupied_housing_units",
    "B25077_001E": "median_home_value",
    "B25088_002E": "median_costs_with_mortgage",
    "B25088_003E": "median_costs_without_mortgage",
    "B25064_001E": "median_gross_rent",
}

print(f"{len(VARIABLES)} variables defined across 4 categories")
print(f"\nCategories:")
print(f"  Age & Sex       → 22 raw variables")
print(f"  Race/Ethnicity  →  7 raw variables")
print(f"  Income/Poverty  →  4 raw variables")
print(f"  Housing         →  7 raw variables")

42 variables defined across 4 categories

Categories:
  Age & Sex       → 22 raw variables
  Race/Ethnicity  →  7 raw variables
  Income/Poverty  →  4 raw variables
  Housing         →  7 raw variables


In [8]:
# Fetch Function 

def fetch_acs(year, geography):
    """
    Fetch raw ACS 5-Year data from Census API.
    
    Parameters:
        year        : int  → ACS year (2024 or 2023)
        geography   : str  → "county", "state", or "nation"
    
    Returns:
        pandas DataFrame with raw Census values
    """
    
    var_string = ",".join(["NAME"] + list(VARIABLES.keys()))
    base       = f"https://api.census.gov/data/{year}/acs/acs5"
    
    if geography == "county":
        url   = (f"{base}?get={var_string}"
                 f"&for=county:{COUNTY_FIPS}"
                 f"&in=state:{STATE_FIPS}"
                 f"&key={CENSUS_API_KEY}")
        label = "Tehama County"
        fips  = TEHAMA_FIPS

    elif geography == "state":
        url   = (f"{base}?get={var_string}"
                 f"&for=state:{STATE_FIPS}"
                 f"&key={CENSUS_API_KEY}")
        label = "California"
        fips  = STATE_FIPS

    elif geography == "nation":
        url   = (f"{base}?get={var_string}"
                 f"&for=us:1"
                 f"&key={CENSUS_API_KEY}")
        label = "United States"
        fips  = "US"

    # Make the request
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    data     = response.json()

    # Parse response
    # Census returns: first row = column names, rest = data
    headers = data[0]
    rows    = data[1:]
    df      = pd.DataFrame(rows, columns=headers)

    # Rename codes to readable names
    df = df.rename(columns=VARIABLES)

    # Convert to numbers
    # Census API returns all values as strings
    # -666666666 is Census code for "not applicable"
    # -999999999 is Census code for "suppressed"
    # We convert these to NaN (not a number = missing)
    for col in VARIABLES.values():
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            df[col] = df[col].where(df[col] > -600000000, other=pd.NA)

    # Add metadata
    df["geography"] = label
    df["geo_type"]  = geography
    df["acs_year"]  = year
    df["fips"]      = fips
    df["fetched_at"]= pd.Timestamp.utcnow().isoformat()

    return df


print(" fetch_acs() function ready")
print("\nUsage examples:")
print("  fetch_acs(2024, 'county')  → Tehama County 2024")
print("  fetch_acs(2024, 'state')   → California 2024")
print("  fetch_acs(2024, 'nation')  → United States 2024")
print("  fetch_acs(2023, 'county')  → Tehama County 2023 (prior year)")

 fetch_acs() function ready

Usage examples:
  fetch_acs(2024, 'county')  → Tehama County 2024
  fetch_acs(2024, 'state')   → California 2024
  fetch_acs(2024, 'nation')  → United States 2024
  fetch_acs(2023, 'county')  → Tehama County 2023 (prior year)


In [11]:
# Fetch Raw Data 
# 2 years × 3 geographies = 6 API calls
# Each call fetches all 40 variables at once

import time

frames = []
errors = []

years      = [2024, 2023]
geographies = ["county", "state", "nation"]

print("Fetching raw Census data...")
print("=" * 45)

for year in years:
    print(f"\nACS Year: {year}")
    for geo in geographies:
        try:
            df = fetch_acs(year, geo)
            frames.append(df)
            print(f"{df['geography'].iloc[0]:20s} → {len(df.columns)} columns")
            time.sleep(0.5)   # be polite to the API
            
        except Exception as e:
            print(f"{geo} ({year}) failed: {e}")
            errors.append(f"{geo} {year}: {e}")

print("\n" + "=" * 45)

if errors:
    print(f"{len(errors)} errors occurred:")
    for err in errors:
        print(f"   {err}")
else:
    print(f"All 6 fetches successful")

# Combine into one master raw DataFrame
raw_df = pd.concat(frames, ignore_index=True)

print(f"\nRaw data summary:")
print(f"  Rows    : {len(raw_df)}")
print(f"  Columns : {len(raw_df.columns)}")
print(f"  Years   : {sorted(raw_df['acs_year'].unique().tolist())}")
print(f"  Places  : {raw_df['geography'].unique().tolist()}")

Fetching raw Census data...

ACS Year: 2024


C:\Users\Vijjeswarapua\AppData\Local\Temp\ipykernel_11204\312629453.py:69: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  df["fetched_at"]= pd.Timestamp.utcnow().isoformat()


Tehama County        → 50 columns


C:\Users\Vijjeswarapua\AppData\Local\Temp\ipykernel_11204\312629453.py:69: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  df["fetched_at"]= pd.Timestamp.utcnow().isoformat()


California           → 49 columns


...

Tehama County        → 50 columns


C:\Users\Vijjeswarapua\AppData\Local\Temp\ipykernel_11204\312629453.py:69: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  df["fetched_at"]= pd.Timestamp.utcnow().isoformat()


California           → 49 columns


C:\Users\Vijjeswarapua\AppData\Local\Temp\ipykernel_11204\312629453.py:69: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  df["fetched_at"]= pd.Timestamp.utcnow().isoformat()


United States        → 49 columns

All 6 fetches successful

Raw data summary:
  Rows    : 6
  Columns : 51
  Years   : [2023, 2024]
  Places  : ['Tehama County', 'California', 'United States']


In [12]:
# Save Raw Data 
# Save exactly what came from the API — no modification
from pathlib import Path

# Define save path 
RAW_DIR = Path("Data/raw/Census")
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Save the complete raw file
raw_path = RAW_DIR / "census_raw_all_geographies.parquet"
raw_df.to_parquet(raw_path, index=False)

# Verify the file was actually created 
file_size = raw_path.stat().st_size

print("RAW DATA SAVED")
print("=" * 50)
print(f"  File    : {raw_path}")
print(f"  Size    : {file_size:,} bytes ({file_size/1024:.1f} KB)")
print(f"  Rows    : {len(raw_df)}")
print(f"  Columns : {len(raw_df.columns)}")

print(f"\nContents:")
for year in sorted(raw_df["acs_year"].unique()):
    for geo in raw_df[raw_df["acs_year"]==year]["geography"].unique():
        subset = raw_df[
            (raw_df["acs_year"]  == year) & 
            (raw_df["geography"] == geo)
        ]
        print(f"{geo} ({year}) → {len(subset)} row")

print(f"\nColumns saved:")
for col in raw_df.columns:
    val = raw_df[raw_df["geography"]=="Tehama County"][col].iloc[0]
    print(f"  {col:<40} {str(val)[:20]}")

RAW DATA SAVED
  File    : Data\raw\Census\census_raw_all_geographies.parquet
  Size    : 32,665 bytes (31.9 KB)
  Rows    : 6
  Columns : 51

Contents:
Tehama County (2023) → 1 row
California (2023) → 1 row
United States (2023) → 1 row
Tehama County (2024) → 1 row
California (2024) → 1 row
United States (2024) → 1 row

Columns saved:
  NAME                                     Tehama County, Calif
  total_population                         65167
  male_population                          32404
  female_population                        32763
  median_age                               40.0
  male_under_5                             1874
  female_under_5                           1882
  male_5_to_9                              2425
  male_10_to_14                            2183
  male_15_to_17                            1443
  female_5_to_9                            2310
  female_10_to_14                          2232
  female_15_to_17                          1347
  male_65_66        